# 数据恢复：独立复算附件

对固定 200 家历史公司、2023-01-03 至 2026-09-30 的 908 个交易日逐项核验。
本附件只读现有原件及 `data-recovery-01/recovered-v5`；不联网、不填财务事实、不改默认模型。
全部字段的分母都是 181,600 个公司交易日，包含退市后的原样本日期，填补估计不计入观测覆盖。
原始文件及后发修订按公开时点区分；本轮补采时间不是历史策略已采集到数据的证明。


In [1]:

import hashlib
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

BASE = Path(r'D:\QuantData\barra-200-v3')
RUN = BASE / 'runs/data-recovery-01'
RECOVERED = RUN / 'recovered-v5'
summary = json.loads((RECOVERED / 'summary.json').read_text(encoding='utf-8'))
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert sha(summary['baseline_panel']) == summary['baseline_sha256']
# The local pickle is trusted only after verifying its frozen original digest.
with Path(summary['baseline_panel']).open('rb') as stream:
    original = pickle.load(stream)['v2']
print('Original SHA-256 unchanged:', summary['baseline_sha256'])


Original SHA-256 unchanged: 32d8c253aa2e27318a3384351889b85fb333825d0fb884ebed7e16a3d63ff4f4


## 1. 从矩阵独立重算覆盖率
使用 NumPy 有限数掩码核对已生成的 CSV，同时检查坐标、无穷值及原非缺失值是否丢失。


In [2]:

rows = []
for item in summary['descriptor_files']:
    assert sha(item['path']) == item['sha256']
    restored = pd.read_parquet(item['path'])
    before = original['descriptors'][item['field']]
    assert restored.index.equals(before.index) and restored.columns.equals(before.columns)
    assert not np.isinf(restored.to_numpy()).any()
    evaluation = (restored.index >= '2023-01-03') & (restored.index <= '2026-09-30')
    a, b = before.loc[evaluation].to_numpy(), restored.loc[evaluation].to_numpy()
    observed_a, observed_b = np.isfinite(a), np.isfinite(b)
    lost = int((observed_a & ~observed_b).sum())
    assert a.size == 181600 and lost == 0
    entry = next(x for x in summary['coverage'] if x['field'] == item['field'])
    assert observed_b.sum() == entry['after_observed']
    rows.append({'field': item['field'], 'before_pct': observed_a.mean()*100,
                 'after_pct': observed_b.mean()*100,
                 'restored_cells': int((~observed_a & observed_b).sum()), 'lost_cells': lost})
display(pd.DataFrame(rows).round(4))


,field,before_pct,after_pct,restored_cells,lost_cells
0,book_to_price,96.2847,96.7847,908,0
1,earnings_yield_ttm,92.8684,96.7093,6975,0
2,earnings_growth_yoy,96.9180,98.2682,2452,0
3,book_leverage,95.9416,96.8822,1708,0
4,revenue_growth_3y,88.9950,97.9262,16219,0
5,profit_growth_3y,90.4752,97.7930,13289,0


## 2. 区分新增版本、解析更正与未准入证据
以公司、字段、所属期、公开可用时间为联合主键。同一事实的后发修订保留独立版本。


In [3]:

keys = ['asset', 'field', 'report_period', 'available_at']
old_facts = pd.read_parquet(BASE / 'runs/main/financial-vintages.parquet')
new_facts = pd.read_parquet(RECOVERED / 'financial-vintages.parquet')
assert not new_facts.duplicated(keys).any()
joined = old_facts.merge(new_facts, on=keys, how='outer', suffixes=('_old', '_new'), indicator=True)
assert not joined._merge.eq('left_only').any()
new_count = int(joined._merge.eq('right_only').sum())
corrected = int((joined._merge.eq('both') & joined.value_old.ne(joined.value_new)).sum())
assert (new_count, corrected) == (1044, 18)
print({'original': len(old_facts), 'recovered': len(new_facts), 'new_vintages': new_count, 'corrected': corrected})
cases = pd.read_csv(RECOVERED / 'annual-case-results.csv', dtype={'asset': str})
display(cases.groupby(['recovery_status', 'classification']).size().rename('cases').to_frame())
assert (cases.recovery_status == 'restored_by_first_affected_date').sum() == 139
assert (cases.recovery_status == 'not_admitted').sum() == 21


{'original': 31149, 'recovered': 32193, 'new_vintages': 1044, 'corrected': 18}


cases
recovery_status                 classification                                
not_admitted                    announcement_not_collected                   2
                                dated_non_disclosure_evidence                6
                                expected_non_disclosure_followup_needed      2
                                pdf_present_extraction_gap                   1
                                post_delisting_source_not_covered           10
restored_by_first_affected_date announcement_not_collected                  40
                                comparative_history_not_loaded              38
                                pdf_present_extraction_gap                  61

## 3. 从逐笔遮蔽预测复算误差
两种方法 × 两种持久缺失方式 × 5%/10%/20% × 2023–2026 年。误差按当时未遮蔽样本的标准差归一化。
这里展示 2023 年开发期、10% 遮蔽；10% 不是已经批准的填补阈值。


In [4]:

masked = pd.read_parquet(RUN / 'imputation-v5/masked-predictions.parquet')
recorded = pd.read_csv(RUN / 'imputation-v5/summary.csv')
keys = ['field', 'method', 'fraction', 'year', 'mask_pattern']
computed = []
for key, part in masked.groupby(keys):
    good = part.dropna(subset=['prediction'])
    error = (good.prediction - good.truth) / good.training_scale
    item = dict(zip(keys, key))
    item.update(normalized_rmse=float(np.sqrt((error**2).mean())),
                prediction_coverage=len(good)/len(part),
                predicted_truth_variance_ratio=float(good.prediction.var()/good.truth.var()))
    computed.append(item)
independent = pd.DataFrame(computed)
comparison = recorded.merge(independent, on=keys, suffixes=('_saved','_computed'), validate='one_to_one')
assert len(comparison) == 96
for name in ['normalized_rmse','prediction_coverage','predicted_truth_variance_ratio']:
    np.testing.assert_allclose(comparison[name+'_saved'], comparison[name+'_computed'], rtol=1e-12, atol=1e-12)
display(recorded[(recorded.year == 2023) & (recorded.fraction == .1)][
    ['field','method','mask_pattern','normalized_rmse','mean_date_rank_correlation','predicted_truth_variance_ratio']].round(4))


,field,method,mask_pattern,normalized_rmse,mean_date_rank_correlation,predicted_truth_variance_ratio
8,profit_growth_3y,industry_median,issuer_full_year,0.7944,0.3268,0.1043
9,profit_growth_3y,industry_median,stratified_63_sessions,0.8273,0.0993,0.1120
32,profit_growth_3y,industry_size_ridge,issuer_full_year,0.8312,0.3624,0.0963
33,profit_growth_3y,industry_size_ridge,stratified_63_sessions,0.8351,0.1013,0.1008
56,revenue_growth_3y,industry_median,issuer_full_year,0.7105,0.4174,0.1715
57,revenue_growth_3y,industry_median,stratified_63_sessions,1.0134,0.0626,0.1050
80,revenue_growth_3y,industry_size_ridge,issuer_full_year,0.7621,-0.0773,0.0576
81,revenue_growth_3y,industry_size_ridge,stratified_63_sessions,1.0948,-0.0904,0.0884


## 4. 重算风险损失与方向
预测期限为 5 个交易日；日收益口径、固定组合及共同资产集合不变。
2024–2026 已被查看，只是回顾性诊断。下面复算损失公式与均值差；区块重采样的完整配置及结果在独立报告中保留。


In [5]:

forecasts = pd.read_parquet(RUN / 'risk-validation-v5/forecasts.parquet')
good = forecasts.dropna(subset=['realized_5d']).copy()
np.testing.assert_allclose(good.variance_5d, good.factor_variance_5d + good.specific_variance_5d, rtol=1e-12)
np.testing.assert_allclose(good.qlike, good.realized_5d**2/good.variance_5d + np.log(good.variance_5d), rtol=1e-12)
risk = json.loads((RUN / 'risk-validation-v5/summary.json').read_text())
differences=[]
for label, years in [('development',[2023]), ('retrospective',[2024,2025,2026])]:
    pivot=good[good.year.isin(years)].pivot(index=['date','family'],columns='model',values='qlike').dropna()
    for name in ['old_input','industry_median','industry_size_ridge']:
        delta=float((pivot[name]-pivot.restored_input).groupby(level='date').mean().mean())
        expected=risk['comparisons'][label]['comparisons'][name]
        assert abs(delta-expected['mean_delta']) < 1e-12
        differences.append({'period':label,'candidate_minus_restored':name,'delta_loss':delta,
                            'simultaneous_95_interval':expected['simultaneous_95_interval']})
display(pd.DataFrame(differences))


,period,candidate_minus_restored,delta_loss,simultaneous_95_interval
0,development,old_input,0.000977,"[-2.151790998569547e-05, 0.00202717174325618]"
1,development,industry_median,0.005127,"[0.0032422573908031187, 0.0066626406349602815]"
2,development,industry_size_ridge,0.005129,"[0.0032429379344040003, 0.006666804081479545]"
3,retrospective,old_input,-0.002885,"[-0.005410544286197848, -8.70797139526139e-05]"
4,retrospective,industry_median,-0.000553,"[-0.003991240185372926, 0.003040409763442341]"
5,retrospective,industry_size_ridge,-0.000553,"[-0.003991240185372926, 0.003040409763442341]"


## 核验结论与边界

- 新增 1,044 个唯一事实版本，更正 18 个原版本；6 个财务描述子没有原观测单元格丢失。
- 139/160 个年度阻塞项在首次受影响日期前恢复；21 项继续保留来源或可靠性限制。
- 填补不能稳定保持个体排序与离散程度，本轮未准入；未把估计值计作已恢复观测。
- 真实数据恢复没有证明风险模型更好：开发期旧减新差异区间跨零；回顾期旧输入损失较低。
- 本附件不替代 62 场景的独立账户/精确现金检查，也不构成新增独立前向证据。
